In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import logging
from sklearn.preprocessing import StandardScaler
import spectral
import random
from skimage import color
import matplotlib.pyplot as plt
import copy
import os

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Set a fixed seed for reproducibility
seed = 10
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Ensure deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

###########################################
# Helper functions
###########################################
def process_rgb(cube, bands, ill, CMFs):
    """
    Converts hyperspectral cube data to RGB using XYZ conversion.
    """
    ill_interp = np.interp(bands, ill[:, 0], ill[:, 1])
    CMFs_interp = np.column_stack([
        np.interp(bands, CMFs[:, 0], CMFs[:, 1]),
        np.interp(bands, CMFs[:, 0], CMFs[:, 2]),
        np.interp(bands, CMFs[:, 0], CMFs[:, 3])
    ])
    sp_tristREF = CMFs_interp * ill_interp[:, None]
    xyz = np.dot(cube, sp_tristREF) / np.sum(sp_tristREF[:, 1], axis=0)
    rgb = color.xyz2rgb(xyz)
    return rgb

###########################################
# Define MLP Model
###########################################
class SimpleMLP(nn.Module):
    def __init__(self, input_size=3, hidden_size=128, output_size=3):
        super(SimpleMLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

###########################################
# Set directories
###########################################
input_directory = '../../data/colorChecker_SG/cubes/'  # Folder with spectral images
output_directory = '../results/plots_iter/'  # Folder to save results
os.makedirs(output_directory, exist_ok=True)

# Load reference data
logging.info("Loading spectral reference data...")
ill = np.loadtxt('../../data/CIE_D65.txt')          
CMFs = np.loadtxt('../../data/CIE2degCMFs_1931.txt')
cube_ref = spectral.open_image('../../data/colorChecker_SG/cubeCC_DigitalSG_REF.hdr')
cube_ref_data = cube_ref.load()
wl_ref = np.array(cube_ref.metadata['wavelength'], dtype=float)

###########################################
# Process each file
###########################################
for file_name in os.listdir(input_directory):
    if file_name.endswith('.hdr'):
        logging.info(f'Processing file: {file_name}')
        cube_path = os.path.join(input_directory, file_name)

        # Load spectral cube
        cube = spectral.open_image(cube_path)
        cube_data = cube.load()
        wl_input = np.array(cube.metadata['wavelength'], dtype=float)

        # Process RGB data
        rgb_input = process_rgb(cube_data, wl_input, ill, CMFs)
        rgb_ref = process_rgb(cube_ref_data, wl_ref, ill, CMFs)

        # Normalize data
        X_flat = rgb_input.reshape(-1, 3)
        Y_flat = rgb_ref.reshape(-1, 3)

        scaler_input = StandardScaler()
        scaler_ref = StandardScaler()

        num_iterations = 4  # Iterative correction steps
        epochs = 500
        batch_size = 4
        alpha = 0.6  # Blending factor for residual learning

        for iteration in range(num_iterations):
            logging.info(f'Iteration {iteration + 1}/{num_iterations} - Training the model...')

            # Dynamically update normalization
            X_norm = scaler_input.fit_transform(X_flat)
            Y_norm = scaler_ref.fit_transform(Y_flat)

            X_train_torch = torch.tensor(X_norm, dtype=torch.float32)
            Y_train_torch = torch.tensor(Y_norm, dtype=torch.float32)

            np.random.seed(seed)
            
            # Split into train/test
            n_pixels = X_norm.shape[0]
            train_size = int(0.8 * n_pixels)
            train_indices = np.random.choice(n_pixels, train_size, replace=False)
            test_indices = np.setdiff1d(np.arange(n_pixels), train_indices)

            X_train = X_train_torch[train_indices]
            Y_train = Y_train_torch[train_indices]
            X_test = X_train_torch[test_indices]
            Y_test = Y_train_torch[test_indices]

            # Initialize model
            model = SimpleMLP()
            optimizer = optim.Adam(model.parameters(), lr=0.001 / (iteration + 1), weight_decay=1e-5)  
            loss_function = nn.MSELoss()

            # Training loop
            for epoch in range(epochs):
                model.train()
                epoch_loss = 0.0
                perm = torch.randperm(X_train.size(0))
                for i in range(0, X_train.size(0), batch_size):
                    batch_indices = perm[i:i+batch_size]
                    X_batch = X_train[batch_indices]
                    Y_batch = Y_train[batch_indices]

                    optimizer.zero_grad()
                    Y_pred = model(X_batch)
                    loss = loss_function(Y_pred, Y_batch)
                    loss.backward()
                    optimizer.step()
                    epoch_loss += loss.item()

                # Validation loss
                model.eval()
                with torch.no_grad():
                    Y_val_pred = model(X_test)
                    val_loss = loss_function(Y_val_pred, Y_test).item()
                
                logging.info(f'Epoch {epoch+1}/{epochs} - Training Loss: {epoch_loss:.4f} - Validation Loss: {val_loss:.4f}')

            # Apply correction
            model.eval()
            with torch.no_grad():
                corrected_flat = model(torch.tensor(X_norm, dtype=torch.float32)).numpy()

            # Residual learning: mix original with corrected
            X_flat = alpha * corrected_flat + (1 - alpha) * X_flat  

        # Final correction
        corrected_rgb = scaler_ref.inverse_transform(X_flat)
        corrected_rgb_image = corrected_rgb.reshape(rgb_ref.shape)

        ###########################################
        # Compute ΔE2000 Error Maps
        ###########################################
        logging.info('Computing ΔE2000 error map...')
        corrected_lab = color.rgb2lab(corrected_rgb_image)
        lab_ref = color.rgb2lab(rgb_ref)

        # Compute ΔE2000 error
        error_map = color.deltaE_ciede2000(lab_ref, corrected_lab)

        # Compute mean & max error
        error_map_flat = error_map.reshape(-1)
        test_error_values = error_map_flat[test_indices]
        mean_error_test = np.mean(test_error_values)
        max_error_test = np.max(test_error_values)

        logging.info(f"Mean ΔE2000 Error: {mean_error_test:.2f}")
        logging.info(f"Max ΔE2000 Error: {max_error_test:.2f}")

        # Get test pixel positions
        test_positions = np.unravel_index(test_indices, lab_ref.shape[:2])

        # Plot error map
        plt.figure(figsize=(8, 6))
        plt.imshow(error_map, cmap='jet', vmin=0, vmax=10)
        plt.colorbar(label='ΔE2000')
        plt.scatter(test_positions[1], test_positions[0], s=2, c='white', label='Test Patches', alpha=0.7)
        plt.title(f'ΔE2000 Error Map - {file_name}')
          # Add text annotations for Mean and Max Error and File name
        file_name_no_extension = os.path.splitext(file_name)[0]
        plt.text(0.5, 1.1, f'Cube: {file_name_no_extension}\nMean Error: {mean_error_test:.2f}\nMax Error: {max_error_test:.2f}', 
                 ha='center', va='bottom', transform=plt.gca().transAxes, fontsize=12, color='black')

        # Adjust layout and save the plot
        plt.tight_layout(pad=3.0)

        # Save plot
        plot_filename = os.path.join(output_directory, f'{os.path.splitext(file_name)[0]}_error_map.png')
        plt.savefig(plot_filename)
        plt.close()

        logging.info(f'Saved error map for {file_name} to {plot_filename}')


2025-03-28 08:07:26,997 - INFO - Loading spectral reference data...
2025-03-28 08:07:26,999 - INFO - Processing file: cubeCC_120f-velvia-f8.hdr
2025-03-28 08:07:26,999 - INFO - Iteration 1/4 - Training the model...
2025-03-28 08:07:27,008 - INFO - Epoch 1/500 - Training Loss: 21.5286 - Validation Loss: 0.5105
2025-03-28 08:07:27,016 - INFO - Epoch 2/500 - Training Loss: 7.2308 - Validation Loss: 0.1921
2025-03-28 08:07:27,023 - INFO - Epoch 3/500 - Training Loss: 3.4636 - Validation Loss: 0.1221
2025-03-28 08:07:27,031 - INFO - Epoch 4/500 - Training Loss: 2.3300 - Validation Loss: 0.0869
2025-03-28 08:07:27,039 - INFO - Epoch 5/500 - Training Loss: 1.7253 - Validation Loss: 0.0674
2025-03-28 08:07:27,046 - INFO - Epoch 6/500 - Training Loss: 1.3559 - Validation Loss: 0.0558
2025-03-28 08:07:27,053 - INFO - Epoch 7/500 - Training Loss: 1.1423 - Validation Loss: 0.0496
2025-03-28 08:07:27,061 - INFO - Epoch 8/500 - Training Loss: 1.0511 - Validation Loss: 0.0457
2025-03-28 08:07:27,068 